# Design of Symmetry-Resolved Real-Space Exact Diagonalization

## Motivation and Scope

The goal of this package is to perform **symmetry-resolved exact diagonalization** (ED) for interacting quantum lattice models, _for both bosonic and fermionic_ — on _arbitrary real-space graphs_. 

The design follows the standard symmetry-resolved ED similar to [`XDiag`](https://github.com/awietek/xdiag), while providing a self-contained, pedagogical Julia implementation.

The key design innovations are:

1. **Bitmask encoding** — every Fock-space configuration $|\bm s\rangle = |n_1,\ldots,n_N\rangle$ (with hard-core constraint $n_i \in \{0,1\}$) is represented by a single `UInt` integer $m = \sum_i n_i 2^{i-1}$, enabling $O(1)$ bitwise operations.

2. **Orbit-stabilizer decomposition** — the many-body Hilbert space is partitioned into orbits under the symmetry group $G$, with each orbit labelled by a canonical representative $|[\bm s]\rangle$ and its stabilizer subgroup data.

3. **Irrep-induced projection** — 1D irreducible representations (irreps) of finite abelian groups are used to construct projectors $P_\chi$ that block-diagonalize the Hamiltonian without ever forming the full matrix.

4. **Two computational modes** — a *matrix mode* that precomputes sparse CSC matrices for fast diagonalization (memory-intensive but fast), and a *matrix-free mode* that computes $H|\psi\rangle$ on-the-fly via multithreaded Lanczos (near-zero memory overhead, $\sim\!1.5\times$ slower).

5. **Unified boson/fermion treatment** — the entire pipeline is statistics-agnostic; fermionic signs (permutation parity and Jordan-Wigner strings) are injected via compile-time multiple dispatch on `Bosonic()`/`Fermionic()` singleton types.

This document provides a self-contained derivation of the underlying theory, explains each algorithmic component, and documents the key data structures. Code snippets are presented in markdown for explanation only: they are not executable.

## Problem Setup

### Many-Body Hilbert Space

Consider $N_e$ **spinless** particles occupying a graph of $N$ vertices. Here **graph vertices represent all internal degrees of freedom flattened into a single index — this includes spatial lattice sites, spin, valley, sublattice, band indices, and even higher-spin or soft-core boson degrees of freedom.** 

Without loss of generality, we work in the **occupation basis** with respect to this graph:

\begin{equation}
    |\bm s\rangle \equiv |n_1,\ldots,n_N\rangle,\qquad n_i \in \{0,1\},\qquad \sum_{i=1}^N n_i = N_e.
\end{equation}

The hard-core constraint $n_i \in \{0,1\}$ is assumed throughout; this is natural for fermions (Pauli exclusion) and is a valid approximation for strongly-interacting bosons. The formalism directly applies to:

- **Fermions**: $c_i^\dagger, c_i$ with $\{c_i,c_j^\dagger\} = \delta_{ij}$.
- **Hard-core bosons**: $b_i^\dagger, b_i$ with $[b_i,b_j^\dagger] = \delta_{ij}$ on different sites, but $(b_i^\dagger)^2 = 0$.

The dimension of the full Hilbert space at fixed $N_e$ is $\binom{N}{N_e}$, which grows combinatorially — the central challenge of ED.

## ⚠️  Understanding `filling_fraction` vs. Community Conventions

A crucial, **frequently-misunderstood** point: the `filling_fraction` parameter in `build_ed_data` is defined as **particles per _flattened_ graph vertex**:

$$\text{filling\_fraction} = \frac{N_{\text{particles}}}{N_{\text{total\_graph\_vertices}}}$$

This is **NOT** the same as the "filling per band" or "filling per site" used in many physics communities. The distinction arises because the ED graph flattens ALL internal degrees of freedom (spin, sublattice, valley, band) into individual vertices.

### Concrete Examples

| Model | Graph Vertices $N$ | Particles $N_e$ | Community "filling" | `filling_fraction` in code |
|---|---|---|---|---|
| **Spin-½ Heisenberg chain** ($L$ sites) | $L$ | $L/2$ | "half-filling" | $\frac{L/2}{L} = 1/2$ |
| **Spinless fermion square** ($L_x\times L_y$) | $L_x L_y$ | $L_x L_y/2$ | "half-filling" | $1/2$ |
| **Spinful Fermi-Hubbard** ($L_x\times L_y$ spatial) | $2 L_x L_y$ (↑+↓) | $L_x L_y$ | "half-filling" (1 $e^-$/site) | $\frac{L_x L_y}{2 L_x L_y} = 1/2$ |
| **Bosonic FCI on Haldane** (2×3 unit cells) | $2\cdot 6 = 12$ (2 sublattices) | $3$ | "$\nu=1/2$ per band" | $3/12 = 1/4$ |
| **Bosonic FCI on Haldane** (2×4 unit cells) | $2\cdot 8 = 16$ | $4$ | "$\nu=1/2$ per band" | $4/16 = 1/4$ |

> **Mnemonic**: Always ask "how many **graph vertices** does my lattice have?" — count every spin, sublattice, and band as a separate vertex. Then compute $N_e / N_{\text{vertices}}$.

### Why This Matters

The "filling per band" convention is natural when thinking about single-particle band structures, but the ED engine works entirely in the **occupation basis of graph vertices** — it has no concept of bands. The `filling_fraction` parameter directly controls how many bits are set in the bitmask representation, and therefore which binomial coefficient $\binom{N}{N_e}$ determines the Hilbert-space dimension.

Using the wrong `filling_fraction` will silently produce results for a **different particle number** than intended, with no error message — because any $0 \le N_e \le N$ is a valid Fock sector. Always double-check this parameter!

### Hamiltonian

As a concrete running example, consider a hard-core bosonic model on a 2D lattice with periodic boundary conditions:

\begin{equation}
    H = \sum_{\langle i,j\rangle} t_{ij}\,b_i^\dagger b_j + \sum_{\langle i,j\rangle} V_{ij}\,n_i n_j,
\end{equation}

where $t_{ij}$ denotes hopping amplitudes and $V_{ij}$ density-density interactions. The framework supports arbitrary short-range terms encoded as lists of `(from_site, to_site, amplitude)` tuples.

More generally, the Hamiltonian takes the form:

\begin{equation}
    H = \underbrace{\sum_{i,j} t_{ij}\,a_i^\dagger a_j}_{\text{bilinear (hopping)}} \;+\; \underbrace{\sum_{i,j} V_{ij}\,n_i n_j}_{\text{density-density}},
\end{equation}

where $a_i^\dagger$ denotes the canonical creation operator (bosonic $b_i^\dagger$ or fermionic $c_i^\dagger$ depending on particle statistics). The model parameters, lattice geometry, and interaction terms are bundled in the `Real_Space_Second_Quantized_Model` struct.

## Bitmask Representation

### Encoding

The hard-core constraint $n_i \in \{0,1\}$ makes each configuration a binary string of length $N$. We encode this as a single unsigned integer:

\begin{equation}
    \boxed{|\bm s\rangle \equiv |n_1,\ldots,n_N\rangle \;\longmapsto\; m = \sum_{i=1}^N n_i\,2^{\,i-1} \;\in\; \{0,1,\ldots,2^N-1\}.}
\end{equation}

Here bit $i-1$ (0-based) corresponds to vertex $i$ (1-based). The type alias is:

```julia
const Mask = UInt
```

This encoding enables $O(1)$ bitwise operations for all physical manipulations:
- **Occupancy test**: `(m & (1 << (i-1))) != 0`
- **Create particle**: `m | (1 << (i-1))`
- **Annihilate particle**: `m & ~(1 << (i-1))`
- **Count particles**: `count_ones(m)` (single CPU instruction)

The bitwise primitives are collected in the `BitWise_Operations` submodule:

```julia
@inline bitmask_of_site(i::Int)::Mask = @fastmath Mask(1) << (i - 1)
@inline occupy_site_for_mask(m::Mask, i::Int)::Mask = @fastmath m | bitmask_of_site(i)
@inline empty_site_for_mask(m::Mask, i::Int)::Mask = @fastmath m & ~bitmask_of_site(i)
@inline is_site_occupied(m::Mask, i::Int)::Bool = @fastmath (m & bitmask_of_site(i)) != 0
@inline is_site_empty(m::Mask, i::Int)::Bool = @fastmath (m & bitmask_of_site(i)) == 0
@inline n_occupied_for_mask(m::Mask) = Base.count_ones(m)
```

All functions are marked `@inline` and `@fastmath` to ensure the compiler emits single CPU instructions with no function-call overhead.

### Why Bitmasks?

The bitmask representation is the foundation of the entire ED pipeline because:

1. **Compact storage**: one 64-bit integer per configuration, regardless of $N \leq 64$ (the practical limit for ED). For $N > 64$, one can generalize to `UInt128` or multi-word bitmasks.

2. **Lexicographic total order**: integer comparison `m < m'` provides a natural canonical ordering of configurations, essential for defining orbit representatives.

3. **Hardware-accelerated operations**: `popcount` (population count, `count_ones`), `ctz` (count trailing zeros), and bitwise AND/OR/XOR are single-cycle CPU instructions, making inner-loop operations blazingly fast.

4. **Gosper's hack compatibility**: enumerating all configurations with exactly $N_e$ particles reduces to iterating integers with a fixed Hamming weight — a well-known combinatorial generation problem solved by Gosper's hack with $O(1)$ per configuration and zero allocation.

## Finite Symmetry Group

### Symmetry Action on Second-Quantized Operators

For a finite system, it is *meaningless* to speak of spontaneous symmetry breaking — that only occurs in the thermodynamic limit. Instead, symmetry is a property of the Hamiltonian $H$ commuting with a unitary representation of a group $G$:

\begin{equation}
    [H, U_g] = 0,\quad \forall g \in G.
\end{equation}

The symmetry action on the creation operators defines how $G$ is realized:

\begin{equation}
    \boxed{U_g\,a_i^\dagger\,U_g^{-1} = \eta_g(i)\,a_{\pi_g(i)}^\dagger},
\end{equation}

where $\pi_g \in S_N$ is a permutation of the $N$ vertices and $\eta_g(i) \in U(1)$ is a site-dependent phase factor. Acting on a Fock state:

\begin{equation}
    U_g|\bm s\rangle = \bigg(\prod_{i\in\text{occ}(\bm s)}\eta_g(i)\bigg)\;
    |n_{\pi_g(1)},\ldots,n_{\pi_g(N)}\rangle.
\end{equation}

In the bitmask picture, this becomes:

\begin{equation}
    U_g\,|m\rangle = \alpha_g(m)\;|m'\rangle,
    \qquad
    \alpha_g(m) = \prod_{i\in\text{occ}(m)}\eta_g(i),
    \qquad
    m' = \sum_{i\in\text{occ}(m)} 2^{\pi_g(i)-1}.
\end{equation}

### Data Structures

Each group element is represented by a `Symmetry_Operation{Group_Label}`:

```julia
struct Symmetry_Operation{Group_Label}
    label::Group_Label              # human-readable label (e.g., (δx,δy) for translations)
    perm::Vector{Int}               # vertex permutation π_g (1-based, length N)
    perm_phases::Vector{ComplexF64}  # site-dependent phase η_g(i)
end
```

The full finite symmetry group is an ordered list of such operations:

```julia
struct Finite_Symmetry_Group
    name::String
    n_site::Int
    operations::Vector{<:Symmetry_Operation}  # ordered list g₁, g₂, …, g_{|G|}
    identity_idx::Int                           # index of identity element
end
```

We typically impose an **order** of the list of group operations in `operations::Vector{<:Symmetry_Operation}`. It is **critical** since all subsequent irrep character vectors are indexed in the same order, enabling $O(1)$ lookup of $\chi(g_i)$ given the group element index $i$.

Typical symmetry groups include:
- **Identity** (no symmetry): $G = \{e\}$, $|G| = 1$.
- **2D lattice translations**: $G = \mathbb Z^{L_1} \times \mathbb Z^{L_2}$, $|G| = L_1 L_2$.
- **Point groups**: $C_3$, $C_4$, $C_6$ rotations.
- **Combined space groups**: direct products of translations and point groups.

### Example: 2D Translation Group

For a rectangular lattice of size $L_1 \times L_2$ with periodic boundary conditions, the translation group is $G = \mathbb Z^{L_1} \times \mathbb Z^{L_2}$. A group element is labelled by the shift vector $\bm\delta = (\delta_1, \delta_2)$ with $\delta_i \in \{0,\ldots,L_i-1\}$. The permutation $\pi_{\bm\delta}$ maps a site at unit cell $\mathbf{R}$ to $\mathbf{R} + \bm\delta$ (modulo the sample size). And because translations do not introduce phases for spinless particles, $\eta_{\bm\delta}(i) \equiv 1$.

```julia
function build_translation_group(lattice::TightBinding.Real_Space_Lattice)::Finite_Symmetry_Group
    n_site = lattice.n_site
    ops = Vector{Symmetry_Operation{Tuple{Int,Int}}}()
    for (δx, δy) in Iterators.product([0:L-1 for L in lattice.sample_size]...)
        perm = Vector{Int}(undef, n_site)
        @inbounds for (i, (cell_int, isub)) in enumerate(lattice.site_list)
            site_shifted = (mod.(cell_int .+ [δx, δy], lattice.sample_size), isub)
            perm[i] = lattice.site_to_index_map[site_shifted]
        end
        push!(ops, Symmetry_Operation((δx, δy), perm))
    end
    return Finite_Symmetry_Group("translations", ops; identity_idx=1)
end
```

## One-Dimensional Irreducible Representations of Finite Abelian Groups

### General Theory

A representation of $G$ is a homomorphism $\rho: G \to GL(V)$. For **finite abelian groups**, all irreducible representations (irreps) are **one-dimensional**. The proof proceeds in three steps:

**Step 1 — Simultaneous diagonalizability.** Since $G$ is abelian, $U_g U_h = U_h U_g$ for all $g,h \in G$. By Schur's lemma, all $U_g$ in an irrep $\rho$ must be proportional to the identity. Thus $\dim V = 1$, and $\rho(g) \in \mathbb{C}^\times \cong U(1)$ (unitarity).

**Step 2 — Structure theorem.** The fundamental theorem of finite abelian groups gives a primary decomposition:

\begin{equation}
    G \cong \bigoplus_{i=1}^r \mathbb{Z}_{p_i^{k_i}},
\end{equation}

or equivalently an invariant factor decomposition $G \cong \bigoplus_{i=1}^r \mathbb{Z}_{d_i}$ with $d_i \mid d_{i+1}$. The number of irreps equals the group order:

\begin{equation}
    |\hat{G}| = |G| = \prod_i p_i^{k_i} = \prod_i d_i.
\end{equation}

**Step 3 — Character formula.** For a cyclic group $\mathbb{Z}_n = \langle a \mid a^n = 1 \rangle$, any 1D irrep $\chi$ is determined by $\chi(a) = e^{2\pi i k/n}$ for some $k \in \{0,\ldots,n-1\}$. For a general element $a^m$:

\begin{equation}
    \boxed{\chi_k(a^m) = e^{2\pi i k m / n}.}
\end{equation}

The direct-sum decomposition extends this to general abelian groups by multiplying phase factors for each cyclic factor.

**Key consequence:** There exists a (non-canonical) isomorphism $G \simeq \hat{G}$ — the group elements and its irreps share the *same* labelling scheme. For translations on a $L_1 \times L_2$ lattice, both group elements $\bm\delta = (\delta_1,\delta_2)$ and irrep labels $\bm k = (k_1,k_2)$ range over the same set $\{0,\ldots,L_1-1\} \times \{0,\ldots,L_2-1\}$.

### Irrep Data Structure

A 1D irrep is stored as the vector of characters evaluated on each group element, in the *same order* as `Finite_Symmetry_Group.operations`:

```julia
struct OneDim_Irrep{Irrep_Label}
    label::Irrep_Label                # e.g., (k1, k2) for momentum sectors
    values::Vector{ComplexF64}        # χ(g₁), χ(g₂), …, χ(g_{|G|})
end
```

The generic type parameter `Irrep_Label` is **distinct** from `Group_Label` in `Symmetry_Operation`:
- `Group_Label` labels a group element (e.g., real-space shift $(\delta_1,\delta_2)$).
- `Irrep_Label` labels an irrep sector (e.g., momentum $\bm k = (k_1,k_2)$).

### Example: 2D Translation Irreps

For $G = \mathbb Z^{L_1} \times \mathbb Z^{L_2}$, the irreps are labelled by crystal momentum $\bm k = (k_1,k_2)$ with $k_i \in \{0,\ldots,L_i-1\}$. With the gauge choice $\chi_{\bm k}(\bm 0) = 1$, the character is:

\begin{equation}
    \boxed{\chi_{\bm k}(\bm\delta) = \exp\!\big[2\pi i\,(k_1\delta_1/L_1 + k_2\delta_2/L_2)\big].}
\end{equation}

```julia
function build_translation_group_irrep_list(G::Finite_Symmetry_Group, lattice::TightBinding.Real_Space_Lattice)
    irrep_list = Vector{OneDim_Irrep{Tuple{Int,Int}}}()
    for k_int in Iterators.product([0:L-1 for L in lattice.sample_size]...)
        χ_list = ComplexF64[]
        for sym_op in G.operations
            cell_int_shift = collect(sym_op.label)  # (δ1, δ2)
            push!(χ_list, cis(2π * sum(cell_int_shift .* collect(k_int) ./ lattice.sample_size)))
        end
        push!(irrep_list, OneDim_Irrep(k_int, χ_list))
    end
    return irrep_list
end
```

The function `cis(x) = exp(im * x)` is a faster Julia built-in.

### Example: $C_3$ Rotation Irreps

For $G = C_3 = \langle R \mid R^3 = 1 \rangle \cong \mathbb{Z}_3$, the group elements are $R^m$ with $m \in \{0,1,2\}$. The irreps are labelled by $l \in \{0,1,2\}$ (angular momentum mod 3):

\begin{equation}
    \chi_l(R^m) = e^{2\pi i l m / 3}.
\end{equation}

Importantly, naive combination of abelian translation and abelian rotation symmetry turns out NO to be abelian in general. Instead, they are **semiproduct** and **non-abelian**, so the 1D-irrep resolving is NOT enough. You do have to consider high-dimensional irrep, with projectors --- this is beyond the scope of our package.

## Orbit-Stabilizer Decomposition

### Definitions

For a configuration $|\bm s\rangle$ (bitmask $m$), we define:

**Orbit:** The set of all configurations reachable by symmetry operations:
\begin{equation}
    \mathrm{Orb}(\bm s) = \{\,U_g|\bm s\rangle \mid g \in G\,\}.
\end{equation}

**Canonical representative:** The _smallest_ bitmask in the orbit (in lexicographic order):
\begin{equation}
    \boxed{|[\bm s]\rangle = \argmin_{|\bm s'\rangle \in \mathrm{Orb}(\bm s)} \mathrm{mask}(|\bm s'\rangle).}
\end{equation}

This is a *gauge choice* — any fixed rule for picking a representative works. Using the smallest bitmask ensures determinism and makes the representative easy to compute.

**Stabilizer subgroup:** The subgroup of $G$ that leaves the representative invariant:
\begin{equation}
    \mathrm{Stab}(\bm s) = \{\,h \in G \mid U_h|[\bm s]\rangle = |[\bm s]\rangle\,\} \subseteq G.
\end{equation}

For each $h \in \mathrm{Stab}(\bm s)$, we record the stabilizer phase:
\begin{equation}
    U_h|[\bm s]\rangle = \alpha_h([\bm s])\,|[\bm s]\rangle,\qquad \alpha_h([\bm s]) \in U(1).
\end{equation}

These phases are *crucial* for symmetry projection — they determine which irreps "see" a given orbit.

### Orbit-Stabilizer Theorem

The fundamental combinatorial identity relating orbit size to stabilizer size:

\begin{equation}
    \boxed{|\mathrm{Orb}(\bm s)| = \frac{|G|}{|\mathrm{Stab}(\bm s)|}.}
\end{equation}

**Proof.** Partition $G$ into left cosets of $\mathrm{Stab}(\bm s)$. For each coset $g\,\mathrm{Stab}(\bm s)$, all elements produce the same state $U_g|[\bm s]\rangle$ (since $U_h$ fixes the representative for $h \in \mathrm{Stab}(\bm s)$). Different cosets produce *distinct* states (otherwise $g_1^{-1}g_2$ would stabilize). Hence the number of distinct states is the number of cosets, $|G|/|\mathrm{Stab}(\bm s)|$.

**Implication:** Knowing only the $N_{\text{orbits}}$ representatives and their stabilizer data suffices to reconstruct the full Hilbert space. The compression ratio for the full sector is:

\begin{equation}
    \frac{N_{\text{orbits}}}{\binom{N}{N_e}} \approx \frac{1}{|G|},
\end{equation}

achieving the optimal $|G|$-fold reduction. Orbits with nontrivial stabilizers (high-symmetry configurations) are slightly over-represented, but this is a small effect for generic fillings.

### The `Symmetry_Orbit_Catalog`

The orbit-stabilizer decomposition is stored in a catalog structure:

```julia
struct Symmetry_Orbit_Catalog
    symmetry_group::Finite_Symmetry_Group
    representative_mask_list::Vector{Mask}           # |[s₁]⟩, |[s₂]⟩, …
    stabilizer_order_list::Vector{Int}               # |Stab(sᵢ)|
    stabilizer_g_indices_list::Vector{Vector{Int}}    # indices of stabilizer group elements
    stabilizer_phases_list::Vector{Vector{ComplexF64}} # α_h([sᵢ]) for each h ∈ Stab(sᵢ)
end
```

The `stabilizer_g_indices_list` and `stabilizer_phases_list` are *parallel* vectors: for orbit $i$, `stabilizer_g_indices_list[i][j]` gives the group element index, and `stabilizer_phases_list[i][j]` gives its eigenvalue phase $\alpha_h([s_i])$.

**Construction algorithm** (in `build_symmetry_orbit_catalog`):
1. Enumerate all bitmasks with exactly $N_e$ particles using Gosper's hack.
2. For each unseen mask $m$, compute its full orbit $\{U_g|m\rangle\}_{g\in G}$.
3. By construction, the first-encountered mask in Gosper order is the orbit minimum — it becomes the representative.
4. Record which group elements leave the representative invariant (the stabilizer) and their phases.
5. Mark all orbit masks as "seen" and skip them in future iterations.
6. Complexity: $O(\binom{N}{N_e} \cdot |G|)$ time, $O(\binom{N}{N_e})$ memory for the seen-set.

## Fast Bitwise Algorithms

### Gosper's Hack: Enumerating Fixed-Weight Bitmasks

To enumerate all configurations with exactly $k$ particles, we need to generate all $N$-bit integers with exactly $k$ bits set. A naive `Combinatorics.combinations(1:N, k)` allocates and encodes each combination — prohibitive for large Hilbert spaces.

**Gosper's hack** iterates *directly* over bitmasks in lexicographic order, generating the next mask with the same Hamming weight in $O(1)$ bitwise operations and *zero allocation*:

```julia
@inline function _gosper_next(x::Mask)::Mask
    c = x & -x            # isolate the rightmost 1-bit
    r = x + c             # add it (forces carry chain)
    return (((r ⊻ x) >> 2) ÷ c) | r   # scatter the displaced bits
end
```

The first mask with $k$ bits set is simply $2^k - 1$. Iteration terminates when the mask exceeds $2^N$.

**Why it works (sketch):** Adding the rightmost 1-bit triggers a carry that clears a block of low-order 1-bits and sets the next 0-bit. The `(r ⊻ x) >> 2` isolates the cleared block, and dividing by `c` right-aligns the scattered 1-bits. The final `| r` reattaches the high bits. The result is the lexicographically next integer with the same popcount.

**Performance:** On a modern CPU, Gosper's hack processes $\sim\!10^8$ masks per second — the bitwise operations are all single-cycle.

### Fast Symmetry Action on a Bitmask

Applying a symmetry operation $U_g$ to a bitmask $m$ requires computing:

\begin{equation}
    m' = \sum_{i\in\text{occ}(m)} 2^{\pi_g(i)-1},\qquad
    \alpha_g(m) = \prod_{i\in\text{occ}(m)} \eta_g(i).
\end{equation}

A naive loop over all $N$ sites scales as $O(N)$. Using bitwise tricks, we can process *only the occupied sites*, scaling as $O(N_e)$ — a substantial win when the system is dilute.

The algorithm repeatedly isolates the rightmost set bit, maps it to its destination, and clears it:

```julia
@inline function apply_operation_to_mask(m::Mask, op::Symmetry_Operation, ::Bosonic)::Tuple{Mask,ComplexF64}
    tmp = m
    new_mask = zero(Mask)
    phase = COMPLEX_ONE                # 1.0 + 0.0im
    @inbounds @fastmath while tmp != 0
        lsb = tmp & -tmp              # isolate rightmost 1-bit
        idx = trailing_zeros(lsb) + 1 # 1-based site index
        new_mask |= Mask(1) << (op.perm[idx] - 1)
        phase *= op.perm_phases[idx]
        tmp ⊻= lsb                     # clear the rightmost 1-bit (XOR)
    end
    return new_mask, phase
end
```

**Key bitwise identities:**
- `x & -x`: isolates the rightmost set bit (two's complement trick).
- `trailing_zeros(x)`: counts trailing zeros (single `TZCNT` CPU instruction) — gives the 0-based bit index.
- `x ⊻= lsb` or `x &= x - 1`: clears the rightmost set bit.

The fermionic version adds a `count_ones` check per occupied site to track the permutation parity; see §Fermionic Particle_Statistics for details.

### Canonical Representative Lookup

Given an *arbitrary* configuration $m$ (not necessarily a representative), find its canonical representative $|[m]\rangle$ and the group element that maps to it:

```julia
function get_canonical_representative(m::Mask, G::Finite_Symmetry_Group, stats::Particle_Statistics)
    repr = m
    best_g = G.identity_idx
    best_amp = COMPLEX_ONE
    @inbounds for (g_idx, g) in enumerate(G.operations)
        shifted, α = apply_operation_to_mask(m, g, stats)
        if shifted < repr
            repr = shifted
            best_g = g_idx
            best_amp = α
        end
    end
    return repr, best_g, best_amp
end
```

This is $O(|G|)$ — acceptable for precomputation, but too slow for the inner loop of Lanczos. For matrix-free mode, we cache results in a `CanonicalMap` (a `Dict{Mask, Tuple{Mask,Int,ComplexF64}}`), achieving amortized $O(1)$ lookup after a one-time pre-scan (see §Matrix-Free Mode).

## Irrep-Induced Projector and Symmetry-Resolved Basis

### Construction of the Projector

Given a 1D irrep $\chi$, define the **group projector**:

\begin{equation}
    \boxed{P_\chi := \frac{1}{|G|}\sum_{g\in G} \chi(g)^*\,U_g.}
\end{equation}

**Proof that $P_\chi$ is a projector.** Using the homomorphism property $\chi(g)\chi(h) = \chi(gh)$ and the group property $U_g U_h = U_{gh}$:

\begin{align}
    P_\chi^2 &= \frac{1}{|G|^2}\sum_{g,h\in G} \chi(g)^* \chi(h)^*\,U_g U_h
            = \frac{1}{|G|^2}\sum_{g,h} \chi(gh)^*\,U_{gh} \nonumber\\
            &= \frac{1}{|G|^2}\sum_{g\in G}\sum_{k\in G} \chi(k)^*\,U_k
            = \frac{1}{|G|}\sum_{g\in G} P_\chi = P_\chi.
\end{align}

Moreover, $[P_\chi, H] = 0$ because each $U_g$ commutes with $H$, and $P_\chi^\dagger = P_\chi$ (unitarity of $\chi$). Thus $P_\chi$ projects onto the symmetry sector corresponding to irrep $\chi$, and the projected Hamiltonian $H_\chi = P_\chi H P_\chi$ is block-diagonal.

### Projection of an Orbit Representative

Apply $P_\chi$ to a canonical representative $|[\bm s]\rangle$. Split the sum over $G$ into cosets of $\mathrm{Stab}(\bm s)$:

\begin{align}
    P_\chi|[\bm s]\rangle
    &= \frac{1}{|G|}\sum_{g\in G} \chi(g)^*\,U_g|[\bm s]\rangle \nonumber\\
    &= \frac{1}{|G|}\sum_{g\in G/\mathrm{Stab}(\bm s)}\;\sum_{h\in\mathrm{Stab}(\bm s)} \chi(gh)^*\,U_{gh}|[\bm s]\rangle \nonumber\\
    &= \frac{1}{|G|}\sum_{g\in G/\mathrm{Stab}(\bm s)} \chi(g)^*\,U_g|[\bm s]\rangle
       \underbrace{\bigg(\sum_{h\in\mathrm{Stab}(\bm s)} \chi(h)^*\,\alpha_h([\bm s])\bigg)}_{\equiv\; S}.
\end{align}

The inner sum $S$ is evaluated using a key lemma:

> **Lemma (Character orthogonality on subgroups).** For a 1D character $\psi: H \to U(1)$ of a finite group $H$:
> - If $\psi$ is trivial ($\psi(h) \equiv 1$), then $\sum_{h\in H} \psi(h) = |H|$.
> - If $\psi$ is non-trivial, then $\sum_{h\in H} \psi(h) = 0$.
>
> **Proof.** If $\psi$ is non-trivial, $\exists h_0 \in H$ with $\psi(h_0) \neq 1$. Under the bijection $h \mapsto h_0 h$, the sum transforms as $S = \sum_h \psi(h_0 h) = \psi(h_0) \sum_h \psi(h) = \psi(h_0) S$. Since $\psi(h_0) \neq 1$, we must have $S = 0$.

Applying the lemma with $H = \mathrm{Stab}(\bm s)$ and $\psi(h) = \chi(h)^* \alpha_h([\bm s])$ gives the **irrep compatibility condition**:

\begin{equation}
    \boxed{P_\chi|[\bm s]\rangle = \begin{cases}
        \displaystyle\frac{|\mathrm{Stab}(\bm s)|}{|G|}\sum_{g\in G/\mathrm{Stab}(\bm s)} \chi(g)^*\,U_g|[\bm s]\rangle, & \text{if } \chi(h) = \alpha_h([\bm s])\;\; \forall h\in\mathrm{Stab}(\bm s),\\[10pt]
        0, & \text{otherwise}.
    \end{cases}}
\end{equation}

An orbit contributes to irrep $\chi$ **iff** the stabilizer phases $\alpha_h$ match the character $\chi(h)$ for every $h \in \mathrm{Stab}(\bm s)$. This is the filtering criterion used to construct the symmetry sector basis.

### Orthonormalization

The projected state $P_\chi|[\bm s]\rangle$ is not normalized. Computing its norm:

\begin{align}
    \big\|P_\chi|[\bm s]\rangle\big\|^2
    &= \frac{|\mathrm{Stab}(\bm s)|^2}{|G|^2} \sum_{g,g'\in G/\mathrm{Stab}(\bm s)} \chi(g)^*\chi(g')\,\alpha_g^*\alpha_{g'}\,
       \langle g\!\cdot\![\bm s]|g'\!\cdot\![\bm s]\rangle \nonumber\\
    &= \frac{|\mathrm{Stab}(\bm s)|^2}{|G|^2} \sum_{g\in G/\mathrm{Stab}(\bm s)} |\alpha_g|^2
     = \frac{|\mathrm{Stab}(\bm s)|^2}{|G|^2} \cdot \frac{|G|}{|\mathrm{Stab}(\bm s)|}
     = \frac{|\mathrm{Stab}(\bm s)|}{|G|}.
\end{align}

Therefore, the **orthonormal projected basis state** is:

\begin{equation}
    \boxed{|\widetilde{[\bm s];\chi}\rangle := \sqrt{\frac{|G|}{|\mathrm{Stab}(\bm s)|}}\;P_\chi|[\bm s]\rangle
    = \sqrt{\frac{|\mathrm{Stab}(\bm s)|}{|G|}}\sum_{g\in G/\mathrm{Stab}(\bm s)} \chi(g)^*\,U_g|[\bm s]\rangle.}
\end{equation}

These states satisfy $\langle\widetilde{[\bm s'];\chi}|\widetilde{[\bm s];\chi}\rangle = \delta_{[\bm s'],[\bm s]}$ and form an orthonormal basis for the symmetry sector.

### The `Symmetry_Sector_Basis`

The symmetry sector basis stores the subset of orbit representatives that satisfy the irrep compatibility condition:

```julia
struct Symmetry_Sector_Basis
    irrep::OneDim_Irrep
    symmetry_group::Finite_Symmetry_Group
    representative_mask_list::Vector{Mask}      # representatives passing the filter
    stabilizer_order_list::Vector{Int}           # |Stab(sᵢ)| for each representative
    repr_to_idx::Dict{Mask,Int}                  # O(1) lookup: repr → index
end
```

The compatibility check is:

```julia
@inline function _is_orbit_compatible(catalog, orbit_idx, irrep; atol=1e-12)::Bool
    gidxs = catalog.stabilizer_g_indices_list[orbit_idx]
    phases = catalog.stabilizer_phases_list[orbit_idx]
    @inbounds for j in eachindex(gidxs)
        !isapprox(irrep.values[gidxs[j]], phases[j]; atol=atol) && return false
    end
    return true
end
```

The construction simply filters the `Symmetry_Orbit_Catalog`:

```julia
function build_symmetry_sector_basis(catalog::Symmetry_Orbit_Catalog, irrep::OneDim_Irrep)
    repr_list = Mask[]
    stab_order_list = Int[]
    for i in eachindex(catalog.representative_mask_list)
        if _is_orbit_compatible(catalog, i, irrep)
            push!(repr_list, catalog.representative_mask_list[i])
            push!(stab_order_list, catalog.stabilizer_order_list[i])
        end
    end
    repr_to_idx = Dict{Mask,Int}(m => idx for (idx, m) in enumerate(repr_list))
    return Symmetry_Sector_Basis(irrep, catalog.symmetry, repr_list, stab_order_list, repr_to_idx)
end
```

The sector dimension is simply `length(basis.representative_mask_list)`. For a generic filling, this is approximately $\binom{N}{N_e}/|G|$, achieving the optimal compression.

## Irrep Projection of an Arbitrary Configuration

### Projecting a Scattered State

When the Hamiltonian $H$ acts on a representative $|[\bm s]\rangle$, it produces a *scattered* configuration $|\bm m\rangle = H|[\bm s]\rangle$ that is generally **no longer a representative**. To compute matrix elements in the orthonormal projected basis, we must project $|\bm m\rangle$ onto the symmetry sector.

The key computational step is `project_to_sector(m, basis)`, which computes $P_\chi|\bm m\rangle$ efficiently. The derivation proceeds in three steps:

**Step 1.** Find the canonical representative of $|\bm m\rangle$ via `get_canonical_representative(m, G, stats)`. This returns:

\begin{equation}
    |[\bm m]\rangle,\qquad g_{\bm m},\qquad \alpha_{g_{\bm m}}
\end{equation}

such that $U_{g_{\bm m}}|\bm m\rangle = \alpha_{g_{\bm m}}|[\bm m]\rangle$, or equivalently $|\bm m\rangle = \alpha_{g_{\bm m}}\,U_{g_{\bm m}^{-1}}|[\bm m]\rangle$.

**Step 2.** Apply the projector $P_\chi$:

\begin{align}
    P_\chi|\bm m\rangle
    &= \frac{1}{|G|}\sum_{g\in G} \chi(g)^*\,U_g|\bm m\rangle
     = \frac{\alpha_{g_{\bm m}}}{|G|}\sum_{g\in G} \chi(g)^*\,U_{g g_{\bm m}^{-1}}|[\bm m]\rangle \nonumber\\
    &= \alpha_{g_{\bm m}}\,\frac{1}{|G|}\sum_{g'\in G} \chi(g' g_{\bm m})^*\,U_{g'}|[\bm m]\rangle
     = \underbrace{\alpha_{g_{\bm m}}\,\chi(g_{\bm m})^*}_{\equiv\;\textsf{coeff}}\;P_\chi|[\bm m]\rangle.
\end{align}

**Step 3.** Express in the orthonormal basis:

\begin{equation}
    \boxed{P_\chi|\bm m\rangle = \textsf{coeff}\cdot\sqrt{\frac{|\mathrm{Stab}(\bm m)|}{|G|}}\;|\widetilde{[\bm m];\chi}\rangle,\qquad
    \textsf{coeff} = \alpha_{g_{\bm m}}\,\chi(g_{\bm m})^*.}
\end{equation}

If $|[\bm m]\rangle$ is not in the sector basis (fails the compatibility condition), then $P_\chi|[\bm m]\rangle = 0$ and the projection returns `nothing`.

### Implementation

The projection code uses the `CanonicalMap` for $O(1)$ representative lookup in matrix-free mode, or falls back to $O(|G|)$ scanning:

```julia
"Fast path: O(1) canonicalization via CanonicalMap"
@inline function _project_fast(m::Mask, basis::Symmetry_Sector_Basis, cmap::CanonicalMap)
    repr, g_idx, α = get_canonical(cmap, m)
    idx = get(basis.repr_to_idx, repr, 0)
    idx == 0 && return nothing
    return (idx, α * conj(basis.irrep.values[g_idx]))
end

"Slow path: O(|G|) canonicalization (used during matrix construction)"
@inline function _project_slow(m::Mask, basis::Symmetry_Sector_Basis, stats::Particle_Statistics)
    repr, g_idx, α = get_canonical_representative(m, basis.symmetry, stats)
    idx = get(basis.repr_to_idx, repr, 0)
    idx == 0 && return nothing
    return (idx, α * conj(basis.irrep.values[g_idx]))
end
```

The returned `(row_idx, coeff)` directly provides the non-zero entries of the Hamiltonian matrix in the orthonormal projected basis.

## Hamiltonian Matrix Construction in the Symmetry Sector

### Matrix Elements

The Hamiltonian matrix element in the orthonormal projected basis is:

\begin{align}
    H_{\bm s',\bm s}^\chi
    &\equiv \langle\widetilde{[\bm s'];\chi}|H|\widetilde{[\bm s];\chi}\rangle \nonumber\\
    &= \sqrt{\frac{|G|}{|\mathrm{Stab}(\bm s)|}}\;
       \langle\widetilde{[\bm s'];\chi}|\,P_\chi H\,|[\bm s]\rangle
       \qquad(\text{since }P_\chi^\dagger = P_\chi,\; P_\chi|\widetilde{[\bm s];\chi}\rangle = |\widetilde{[\bm s];\chi}\rangle) \nonumber\\
    &= \sqrt{\frac{|G|}{|\mathrm{Stab}(\bm s)|}}\;
       \langle\widetilde{[\bm s'];\chi}|\,P_\chi\,|\bm m\rangle
       \qquad(\text{where }|\bm m\rangle \equiv H|[\bm s]\rangle) \nonumber\\
    &= \sqrt{\frac{|G|}{|\mathrm{Stab}(\bm s)|}}\;
       \textsf{coeff}\cdot\sqrt{\frac{|\mathrm{Stab}(\bm m)|}{|G|}}\;
       \langle\widetilde{[\bm s'];\chi}|\widetilde{[\bm m];\chi}\rangle \nonumber\\
    &= \textsf{coeff}\cdot\sqrt{\frac{|\mathrm{Stab}(\bm m)|}{|\mathrm{Stab}(\bm s)|}}\;
       \delta_{[\bm s'],[\bm m]}.
\end{align}

**Key result:** The matrix element depends on (i) the raw scattering amplitude of $H$ (absorbed into $\mathsf{coeff}$ via the projection), and (ii) a rescaling factor $\sqrt{|\mathrm{Stab}(\bm m)|/|\mathrm{Stab}(\bm s)|}$ that corrects for the different stabilizer sizes of the initial and final representatives.

For diagonal terms, i.e., density-density terms in the occupation basis, $|\bm m\rangle = V n_i n_j|[\bm s]\rangle = V|[\bm s]\rangle$ (the representative is unchanged), so $\mathsf{coeff} = 1$, $|\mathrm{Stab}(\bm m)| = |\mathrm{Stab}(\bm s)|$, and $H_{\bm s,\bm s}^\chi = V$ — no projection overhead.

### Sparse Matrix Construction Algorithm

The function `build_ed_Hamiltonian_symmetry_block` iterates over all representative masks in the sector basis and constructs the sparse CSC matrix:

```julia
function build_ed_Hamiltonian_symmetry_block(
    basis::Symmetry_Sector_Basis,
    bilinear_terms::Vector{<:Tuple{Int,Int,<:Number}},
    density_terms::Vector{<:Tuple{Int,Int,<:Number}},
    particle_statistics::Particle_Statistics,
)::SparseMatrixCSC{ComplexF64,Int}
    sector_dim = length(basis.representative_mask_list)

    Is, Js, Vs = Int[], Int[], ComplexF64[]
    # ... (preallocate with estimated nnz)

    @inbounds for (col, repr_mask) in enumerate(basis.representative_mask_list)
        stab_col = basis.stabilizer_order_list[col]

        # Diagonal: density-density
        H_diag = zero(ComplexF64)
        for (i, j, V) in density_terms
            if is_site_occupied(repr_mask, i) && is_site_occupied(repr_mask, j)
                H_diag += V
            end
        end
        push!(Is, col); push!(Js, col); push!(Vs, H_diag)

        # Off-diagonal: hopping
        for (i_from, i_to, t) in bilinear_terms
            if is_site_occupied(repr_mask, i_from) && is_site_empty(repr_mask, i_to)
                new_mask = empty_site_for_mask(repr_mask, i_from)
                new_mask = occupy_site_for_mask(new_mask, i_to)

                proj = _project_slow(new_mask, basis, particle_statistics)
                proj === nothing && continue
                row, coeff = proj
                stab_row = basis.stabilizer_order_list[row]

                hop_phase = hopping_phase_for_stats(particle_statistics, repr_mask, i_from, i_to)
                H_elem = t * hop_phase * coeff * sqrt(stab_row / stab_col)

                push!(Is, row); push!(Js, col); push!(Vs, H_elem)
            end
        end
    end

    H = sparse(Is, Js, Vs, sector_dim, sector_dim)
    dropzeros!(H)
    return H
end
```

The sparsity of the resulting matrix is approximately $O(\text{const} \times N / \text{sector\_dim})$ — extremely sparse for large sectors, making Arpack's iterative eigensolver highly efficient.

### Matrix-Free Mode: On-the-Fly $H|\psi\rangle$

In matrix-free mode, the sparse matrix is never stored. Instead, `apply_hamiltonian!` computes $y = H \cdot x$ directly:

```julia
function apply_hamiltonian!(y::Vector{ComplexF64}, x::Vector{ComplexF64},
    basis::Symmetry_Sector_Basis,
    bilinear_terms, density_terms, cmap::CanonicalMap,
    y_threads::Vector{Vector{ComplexF64}})     # per-thread accumulation buffers

    # Zero out thread-local buffers
    @inbounds for tid in 1:Threads.maxthreadid()
        fill!(y_threads[tid], 0)
    end

    Threads.@threads :static for col in 1:n
        tid = Threads.threadid()
        yt = y_threads[tid]
        x_col = x[col]
        x_col == 0 && continue

        repr_mask = basis.representative_mask_list[col]
        stab_col = basis.stabilizer_order_list[col]
        inv_sqrt_stab_col = 1.0 / sqrt(stab_col)

        # Diagonal
        H_diag = zero(ComplexF64)
        for (i, j, V) in density_terms
            if is_site_occupied(repr_mask, i) && is_site_occupied(repr_mask, j)
                H_diag += V
            end
        end
        yt[col] += H_diag * x_col

        # Off-diagonal
        for (i_from, i_to, t) in bilinear_terms
            if is_site_occupied(repr_mask, i_from) && is_site_empty(repr_mask, i_to)
                new_mask = empty_site_for_mask(repr_mask, i_from)
                new_mask = occupy_site_for_mask(new_mask, i_to)

                proj = _project_fast(new_mask, basis, cmap)
                proj === nothing && continue
                row, coeff = proj
                stab_row = basis.stabilizer_order_list[row]
                hop_phase = hopping_phase_for_stats(cmap.particle_statistics, repr_mask, i_from, i_to)
                H_elem = t * hop_phase * coeff * sqrt(stab_row) * inv_sqrt_stab_col
                yt[row] += H_elem * x_col
            end
        end
    end

    # Reduce thread-local buffers → y
    fill!(y, 0)
    @inbounds for tid in 1:Threads.maxthreadid()
        yt = y_threads[tid]
        for i in 1:n
            y[i] += yt[i]
        end
    end
    return y
end
```

**Key optimization — `CanonicalMap`:** Before starting Lanczos, `populate_canonical_map!` pre-scans all reachable configurations from every representative mask and caches their canonical representatives. This reduces the projection step from $O(|G|)$ to $O(1)$ hash-table lookup, making matrix-free mode only $\sim\!1.5\times$ slower than the explicit sparse matrix — a design directly inspired by XDiag.

The `CanonicalMap` is a thread-safe read-only dictionary shared across all threads:

```julia
struct CanonicalMap
    symmetry_group::Finite_Symmetry_Group
    particle_statistics::Particle_Statistics
    cache::Dict{Mask,Tuple{Mask,Int,ComplexF64}}   # scattered → (repr, g_idx, α)
end
```

**Thread safety:** Each thread accumulates into its own `y_threads[tid]` buffer, then results are summed at the end — no locks, no atomics, fully race-free.

## Fermionic Particle_Statistics

### Fock Space Ordering Convention

For **spinless fermions** on $N$ vertices, the Fock state is defined with a fixed ordering convention:

\begin{equation}
    \boxed{|\bm s\rangle \equiv |n_1,\ldots,n_N\rangle := \prod_{i=1}^N (c_i^\dagger)^{n_i}\,|0\rangle,}
\end{equation}

where the product runs in **increasing order** $i = 1,2,\ldots,N$. This single convention determines *all* subsequent sign factors — both the permutation parity in symmetry operations and the Jordan-Wigner string in hopping.

Crucially, the **bitmask representation $m = \sum_i n_i 2^{i-1}$ remains unchanged** from the bosonic case. The *only* difference is that physical amplitudes now carry additional $\pm 1$ factors.

### Fermionic Sign in Symmetry Operations

The symmetry action on fermionic operators is defined identically to the bosonic case:

\begin{equation}
    U_g\,c_i^\dagger\,U_g^{-1} = \eta_g(i)\,c_{\pi_g(i)}^\dagger.
\end{equation}

Acting on a Fock state:

\begin{align}
    U_g|\bm s\rangle
    &= \prod_{i\in\mathrm{occ}(\bm s)} \bigl(\eta_g(i)\,c_{\pi_g(i)}^\dagger\bigr)\,|0\rangle \nonumber\\
    &= \Bigl(\prod_{i\in\mathrm{occ}(\bm s)}\eta_g(i)\Bigr)\;
       \mathrm{sgn}\bigl(\pi_g|_{\mathrm{occ}(\bm s)}\bigr)\;
       |\bm s'\rangle.
\end{align}

The new factor $\mathrm{sgn}(\pi_g|_{\mathrm{occ}(\bm s)})$ is the **parity of the permutation restricted to occupied sites**. It arises because, after applying $U_g$, the creation operators $c_{\pi_g(i)}^\dagger$ are no longer in the canonical increasing order; reordering them to match convention (1) introduces a factor of $(-1)^{\text{number of inversions}}$.

**Concretely:** let the occupied sites be $i_1 < i_2 < \cdots < i_{N_e}$ with images $j_a = \pi_g(i_a)$. Then:

\begin{equation}
    \mathrm{sgn}\bigl(\pi_g|_{\mathrm{occ}(\bm s)}\bigr) = (-1)^{\#\{(a,b) \,:\, a < b,\; j_a > j_b\}}.
\end{equation}

Each pair $(a,b)$ with $a < b$ but $j_a > j_b$ contributes a factor $-1$ because the two creation operators must be swapped to restore the canonical order.

### Efficient Bitwise Computation of the Permutation Sign

The inversion count can be computed *inline* during the same bit-loop that builds the transformed mask, with only a single `count_ones` per occupied site. Recall that the algorithm processes occupied sites from *rightmost* (smallest $i$) to *leftmost* (largest $i$). At step $i$, the destinations of all previously processed (smaller) occupied sites are already set in `new_mask`.

Just *before* setting bit $j-1$ (where $j = \pi_g(i)$), the bits already set in `new_mask` at positions $\geq j$ correspond to earlier-processed occupied sites whose destination exceeds $j$ — precisely the inversions involving site $i$:

\begin{equation}
    (\text{inversions from site }i) = \mathtt{count\_ones}\bigl(\mathtt{new\_mask} \gg (j-1)\bigr).
\end{equation}

The total fermionic sign is therefore:

\begin{equation}
    \mathrm{sgn}\bigl(\pi_g|_{\mathrm{occ}(m)}\bigr) = \prod_{i\in\mathrm{occ}(m)} (-1)^{\mathtt{count\_ones}(\mathtt{new\_mask}\,\gg\,(\pi_g(i)-1))}.
\end{equation}

The implementation adds this logic to the bosonic `apply_operation_to_mask!`:

```julia
@inline function apply_operation_to_mask(m::Mask, op::Symmetry_Operation, ::Fermionic)::Tuple{Mask,ComplexF64}
    tmp = m
    new_mask = zero(Mask)
    phase = COMPLEX_ONE
    parity = Int8(1)
    @inbounds @fastmath while tmp != 0
        lsb = tmp & -tmp
        idx = trailing_zeros(lsb) + 1
        p = op.perm[idx]
        if isodd(count_ones(new_mask >> p))    # <-- fermionic addition
            parity = -parity
        end
        new_mask |= Mask(1) << (p - 1)
        phase *= op.perm_phases[idx]
        tmp ⊻= lsb
    end
    return new_mask, phase * parity
end
```

The overhead is a single `POPCNT` instruction per occupied site — negligible relative to the existing bit operations.

### Fermionic Sign in Hopping — The Jordan-Wigner String

Applying a hopping term $c_j^\dagger c_i$ to a Fock state $|\bm s\rangle$ with $n_i = 1$, $n_j = 0$ (assume $i < j$):

1. **Annihilation by $c_i$:** The operator $c_i$ anticommutes past every creation operator with index $< i$, picking up $(-1)^{\sum_{k<i} n_k}$.

2. **Creation by $c_j^\dagger$:** After $c_i$ has acted (removing $c_i^\dagger$), $c_j^\dagger$ must be inserted at its canonical position. It anticommutes past all creation operators with $k < j$, $k \neq i$, yielding $(-1)^{\sum_{k<j,\,k\neq i} n_k}$.

The total sign is:

\begin{align}
    (-1)^{\sum_{k<i} n_k + \sum_{k<j,\,k\neq i} n_k}
    &= (-1)^{2\sum_{k<i} n_k + \sum_{i<k<j} n_k}
     = (-1)^{\sum_{i<k<j} n_k}.
\end{align}

Generalizing to arbitrary $i \neq j$:

\begin{equation}
    \boxed{c_j^\dagger c_i\,|\bm s\rangle = (-1)^{\displaystyle\sum_{k\,=\,\min(i,j)+1}^{\max(i,j)-1} n_k}\;
    |\bm s; i\!\to\! j\rangle.}
\end{equation}

This is the standard **Jordan-Wigner string** sign — it depends only on the occupied sites *strictly between* $i$ and $j$.

### Bitmask Implementation

The sum of occupancies between sites $\mathtt{lo}$ and $\mathtt{hi}$ is computed with a single bitmask AND followed by `count_ones`:

```julia
@inline function hopping_phase_for_stats(::Fermionic, m::Mask, i_from::Int, i_to::Int)::ComplexF64
    i_from == i_to && return COMPLEX_ONE
    lo = min(i_from, i_to)
    hi = max(i_from, i_to)
    # mask_between = bits lo … hi-2 set (0-based).  Example: lo=2,hi=5 → bits 2,3 set → 0b1100₂
    between = hi - lo <= 1 ? zero(Mask) : (((one(Mask) << (hi - lo - 1)) - one(Mask)) << lo)
    return isodd(count_ones(m & between)) ? -COMPLEX_ONE : COMPLEX_ONE
end
```

For bosons, the corresponding function simply returns `+1`:

```julia
@inline hopping_phase_for_stats(::Bosonic, ::Mask, ::Int, ::Int)::ComplexF64 = COMPLEX_ONE
```

### Multiple Dispatch Design

Julia's multiple dispatch on the singleton types `Bosonic()` and `Fermionic()` resolves the statistics at **compile time** — there is zero branching overhead in inner loops. The algebraic data type is defined via `MLStyle.@data`:

```julia
MLStyle.@data Particle_Statistics begin
    Bosonic()
    Fermionic()
end
```

Both statistics types are embedded in the second-quantized model:

```julia
struct Real_Space_Second_Quantized_Model{T} <: Real_Space_Second_Quantized_Model
    params::Dict
    lattice::TightBinding.Real_Space_Lattice
    tb_model::TightBinding.Real_Space_TightBinding_Model
    particle_statistics::Particle_Statistics                          # Bosonic() or Fermionic()
    bilinear_terms::Vector{Tuple{Int,Int,T}}         # hopping (a†_i a_j)
    density_density_terms::Vector{Tuple{Int,Int,T}}  # n_i n_j
end
```

For hot inner loops (`apply_operation_to_mask!`, called millions of times), direct dispatch on `::Bosonic` / `::Fermionic` is used. For higher-level control flow, `MLStyle.@match` provides readable dispatching:

```julia
@match particle_statistics begin
    Bosonic()   => _build_chunk_bosonic(...)
    Fermionic() => _build_chunk_fermionic(...)
end
```

### What Stays Unchanged

The structure of the ED pipeline is completely preserved between bosons and fermions:
- **Diagonal density-density terms:** $n_i n_j$ are products of occupation numbers — identical for both statistics.
- **Irrep-induced projector:** The fermionic permutation sign is already absorbed into the stabilizer phases $\alpha_h([\bm s])$ via the modified symmetry action; the projector definition $P_\chi = \frac{1}{|G|}\sum_g \chi(g)^* U_g$ is statistics-agnostic.
- **Orbit-stabilizer decomposition:** The *set* of orbit representatives depends only on the lexicographic order of bitmasks, which is independent of particle statistics.
- **Irrep compatibility condition:** $\chi(h) = \alpha_h([\bm s])$ for all $h \in \mathrm{Stab}(\bm s)$ remains the correct filtering criterion, since the stabilized phases $\alpha_h$ are correctly computed by the fermionic symmetry action.
- **ED diagonalization:** Arpack/KrylovKit only see the finished sparse matrix or the linear operator; they are completely agnostic to the underlying statistics.

## HPC and Parallelism

### Distributed Hamiltonian Construction

For large sectors where the sparse matrix is too large to build on a single node, `build_ed_Hamiltonian_symmetry_block_distributed` uses `Distributed.pmap` to partition representative masks across workers. Each worker processes a chunk independently, producing `(Is, Js, Vs)` triplets, which are then concatenated on the master:

```julia
nw = nworkers()
chunk_size = cld(sector_dim, nw)
col_ranges = [r[1]:r[end] for r in Iterators.partition(1:sector_dim, chunk_size)]

results = pmap(pool, col_ranges) do rng
    # ... build (Is, Js, Vs) for this chunk ...
    return (Is, Js, Vs)
end

Is = reduce(vcat, getindex.(results, 1))
Js = reduce(vcat, getindex.(results, 2))
Vs = reduce(vcat, getindex.(results, 3))
H = sparse(Is, Js, Vs, sector_dim, sector_dim)
```

Workers are loaded once with `ensure_distributed_workers_loaded!()`, which evaluates `using RealSpace_ExactDiagonalization` on each worker and sets `BLAS.set_num_threads(1)` to avoid oversubscription.

### Distributed Matrix-Free $H|\psi\rangle$

For matrix-free mode on HPC clusters, `apply_hamiltonian_distributed!` partitions column indices across workers. Each worker receives a chunk of columns, computes its partial contribution to $y$, and the results are summed on the master:

```julia
function apply_hamiltonian_distributed!(y, x, basis, bilinear_terms, density_terms, cmap)
    # ... partition columns across workers ...
    results = pmap(pool, col_ranges) do rng
        y_part = zeros(ComplexF64, n)
        # ... compute y_part for columns in rng ...
        return y_part
    end
    fill!(y, 0)
    for yp in results
        y .+= yp
    end
    return y
end
```

The `CanonicalMap` is broadcast to all workers so they can resolve canonical representatives without network round-trips.

### Thread-Level Parallelism (Shared Memory)

Within a single node, `Threads.@threads :static` distributes columns across CPU cores. The `static` schedule ensures contiguous column assignment for cache locality. Per-thread accumulation buffers eliminate races:

```julia
y_threads = [zeros(ComplexF64, n) for _ in 1:Threads.maxthreadid()]

Threads.@threads :static for col in 1:n
    tid = Threads.threadid()
    yt = y_threads[tid]      # thread-private buffer
    # ... accumulate into yt ...
end

# Reduce
for tid in 1:Threads.maxthreadid()
    y .+= y_threads[tid]
end
```

This achieves near-ideal scaling up to $\sim\!40$ cores. For diagonalization, `BLAS.set_num_threads(nprocs())` is temporarily set to leverage multi-threaded LAPACK, then restored to 1 afterward.

## Complete ED Pipeline

### Data Flow

The top-level data structure `Symmetry_Resolved_ED_Data` bundles all components:

```julia
mutable struct Symmetry_Resolved_ED_Data
    model::Real_Space_Second_Quantized_Model
    n_filled::Int
    filling_fraction::Rational{Int}
    symmetry_group::Finite_Symmetry_Group
    irrep_list::Vector{OneDim_Irrep}
    orbit_catalog::Symmetry_Orbit_Catalog
    sector_dims::Vector{Int}
    ed_scan_res::Dict{Int,Tuple{Vector{Float64},Matrix{ComplexF64}}}
end
```

The complete pipeline proceeds as:

1. **Define model** — construct `Real_Space_Second_Quantized_Model` with lattice geometry, hopping and interaction terms, and particle statistics.
2. **Build symmetry group** — construct `Finite_Symmetry_Group` (e.g., translations, rotations) and its irreps.
3. **Build orbit catalog** — `build_symmetry_orbit_catalog(model, n_filled, symmetry, particle_statistics)` partitions the Hilbert space into orbits.
4. **Scan irreps** — for each $\chi \in \hat{G}$:
   - Build `Symmetry_Sector_Basis` by filtering the orbit catalog.
   - Choose **matrix mode** (`ed_scan_at_irrep_matrix!`) or **matrix-free mode** (`ed_scan_at_irrep_matrixfree!`).
   - Diagonalize with Arpack (matrix) or KrylovKit (matrix-free).
5. **Post-process** — analyze spectra, compute correlators, plot results.

### Memory Considerations

- **Matrix mode:** Stores the full sparse CSC matrix for each sector. Memory scales as $O(\text{sector\_dim} \times \text{coordination})$. Suitable for sector dimensions up to $\sim\!10^5$.
- **Matrix-free mode:** Only stores basis vectors ($O(\text{sector\_dim})$) and the `CanonicalMap` ($O(\text{unique reachable configs})$). Suitable for sector dimensions up to $\sim\!10^7$.
- **Orbit catalog:** $O(\text{sector\_dim})$ — the dominant memory cost independent of mode.

### Checkpoint-Restore

Long-running ED scans support checkpointing via JLD2:

```julia
function save_checkpoint(ed_data::Symmetry_Resolved_ED_Data, filepath::String)
    jldsave(filepath; ed_data=ed_data)
end

function load_checkpoint(filepath::String)::Symmetry_Resolved_ED_Data
    return load(filepath, "ed_data")
end
```

This enables fault-tolerant HPC runs: if a job is preempted, it resumes from the last checkpoint with all previously computed sectors intact.

## Summary of Key Formulas

| Quantity | Formula |
|---|---|
| Bitmask encoding | $m = \sum_i n_i 2^{i-1}$ |
| Symmetry action | $U_g\|m\rangle = \alpha_g(m)\,\|m'\rangle$, $\;m' = \sum_{i\in\mathrm{occ}(m)} 2^{\pi_g(i)-1}$ |
| Orbit size | $\|\mathrm{Orb}(\bm s)\| = \|G\| / \|\mathrm{Stab}(\bm s)\|$ |
| 1D irrep character (cyclic) | $\chi_k(a^m) = e^{2\pi i k m / n}$ |
| Irrep projector | $P_\chi = \frac{1}{\|G\|}\sum_g \chi(g)^* U_g$ |
| Irrep compatibility | $\chi(h) = \alpha_h([\bm s])\;\forall h\in\mathrm{Stab}(\bm s)$ |
| Orthonormal projected state | $\|\widetilde{[\bm s];\chi}\rangle = \sqrt{\frac{\|\mathrm{Stab}(\bm s)\|}{\|G\|}}\sum_{g\in G/\mathrm{Stab}(\bm s)} \chi(g)^* U_g\|[\bm s]\rangle$ |
| Projection coefficient | $\mathsf{coeff} = \alpha_{g_{\bm m}}\,\chi(g_{\bm m})^*$ |
| Hamiltonian matrix element | $H_{\bm s',\bm s}^\chi = \mathsf{coeff}\cdot\sqrt{\frac{\|\mathrm{Stab}(\bm m)\|}{\|\mathrm{Stab}(\bm s)\|}}\;\delta_{[\bm s'],[\bm m]}$ |
| Fermionic symmetry sign | $\mathrm{sgn}(\pi_g\|_{\mathrm{occ}}) = \prod_i (-1)^{\mathtt{count\_ones}(\mathtt{new\_mask}\,\gg\,(\pi_g(i)-1))}$ |
| Jordan-Wigner sign | $(-1)^{\sum_{k=\min(i,j)+1}^{\max(i,j)-1} n_k}$ |

The implementation faithfully realizes all of these formulas in a high-performance, statistics-agnostic Julia codebase suitable for both pedagogical exploration and production HPC runs.

## Examples and Usage

The `examples/` folder contains three self-contained, ready-to-run scripts that demonstrate the full ED pipeline on three canonical models. Each example is designed to be pedagogical — it builds the lattice via TightBinding, constructs the second-quantised model, sets up translation symmetry, runs symmetry-resolved ED, and reports the lowest eigenvalues. Users can copy and adapt these scripts for their own models.

All examples use the [`TightBinding`](https://github.com/GrothendieckUniverse/TightBinding) package for lattice and tight-binding model construction, ensuring a clean, declarative workflow.

### Example 1: Spin-½ Heisenberg Chain

The 1D antiferromagnetic Heisenberg model with periodic boundary conditions:

\begin{equation}
    H = J \sum_{\langle i,j\rangle} \bm S_i \cdot \bm S_j, \qquad J > 0 \text{ (antiferromagnetic)}.
\end{equation}

Using the **Matsubara-Matsuda** mapping to hard-core bosons:
\begin{equation}
    S^z_i = n_i - \tfrac12,\qquad S^+_i = b^\dagger_i,\qquad S^-_i = b_i,
\end{equation}

the spin-spin interaction expands to:

\begin{equation}
    \boxed{\bm S_i \cdot \bm S_j = \cdots = n_i n_j + \tfrac12\big(b^\dagger_i b_j + \mathrm{h.c.}\big) - \tfrac12 n_i - \tfrac12 n_j + \tfrac14}.
\end{equation}

Summing the constant contributions over all bonds at half-filling ($N_e = N/2$): $\sum_{\langle i,j\rangle}(-\frac12 n_i - \frac12 n_j) = -\sum_i n_i = -N_e$, and $\sum_{\langle i,j\rangle}\frac14 = N/4$, giving a net constant $-JN/4$.  This is absorbed by adding on-site density terms $(i,i,-J/2)$ for each site $i$:

\begin{equation}
    \sum_{i} (-\tfrac{J}{2})\,n_i = -\frac{J}{2}\,N_e = -\frac{JN}{4}.
\end{equation}

Consequently, the code's Hamiltonian directly equals the physical spin Hamiltonian — no external constant-shift bookkeeping is needed.

The model is solved in the $S^z_{\rm total}=0$ sector ($N_e = N/2$, half-filling).  Translation symmetry $\mathbb Z^N$ provides $N$ momentum sectors.

**Key features demonstrated:**
- Construction of a 1D chain lattice with PBC via `TightBinding.initialize_real_space_lattice`
- Complete Matsubara-Matsuda mapping with on-site $(i,i,-J/2)$ terms absorbing all constants
- `add_hopping_term!`'s automatic translation expansion — only one representative bond needed
- Translation symmetry resolution into 1D momentum sectors
- Comparison with the known Bethe-ansatz result $E_0/N = -\ln 2 + 1/4 \approx -0.443147$

```bash
julia --project=. examples/spin_heisenberg_chain.jl
```

### Example 2: Bosonic Fractional Chern Insulator

The Haldane honeycomb model with hard-core bosons at $\nu = 1/2$ per band, following D.N. Sheng et al. (PRL **107**, 146803, 2011):

\begin{equation}
    H = \sum_{\langle i,j\rangle} t_{ij}\,b^\dagger_i b_j + \sum_{\langle\!\langle i,j\rangle\!\rangle} t'_{ij}\,b^\dagger_i b_j + \sum_{\langle\!\langle\!\langle i,j\rangle\!\rangle\!\rangle} t''_{ij}\,b^\dagger_i b_j,
\end{equation}

with complex next-nearest-neighbour hoppings $t' = -0.60\,e^{\pm i\phi}$ ($\phi = 0.4\pi$) that break time-reversal symmetry. The lattice has 2 sublattices; at $2\times 3$ unit cells there are 12 sites. With 3 particles (half-filling of the lower band), the full Hilbert space dimension is $\binom{12}{3} = 220$, which is reduced to 38–45 orbits per momentum sector by the $\mathbb Z^2 \times \mathbb Z^3$ translation group.

**Expected result:** Two nearly degenerate ground states at $E \approx -7.1638$ (k = (0,0)) and $E \approx -7.1634$ (k = (1,0)), with a topological splitting $\Delta E \approx 4.3 \times 10^{-4}$ — the hallmark of a $\nu = 1/2$ Laughlin-type FCI on a torus.

```bash
julia --project=. examples/boson_fci_haldane.jl
```

**Key features demonstrated:**
- Construction of a multi-sublattice 2D lattice with complex hopping phases
- Translation symmetry with $|G| = L_1 \times L_2 = 6$
- Benchmark against published DMRG results
- Both matrix and matrix-free ED modes available

### Example 3: Spinful Fermi-Hubbard Model

The canonical model of strongly correlated electrons on a square lattice:

\begin{equation}
    H = -t \sum_{\langle i,j\rangle,\sigma} \big(c^\dagger_{i\sigma} c_{j\sigma} + \text{h.c.}\big) + U \sum_i n_{i\uparrow} n_{i\downarrow},
\end{equation}
with $t = 1$, $U = 8$, and periodic boundary conditions. 

Importantly, the **spin degrees of freedom are handled by the _flattened-graph approach_**: 
- Each spatial site $i$ generates two graph vertices — $i_\uparrow$ (vertex $2i-1$) and $i_\downarrow$ (vertex $2i$) — in interleaved order. 

This doubling gives $2 \times L_x L_y$ vertices; at half-filling per spin ($N_\uparrow = N_\downarrow = L_x L_y/2$), the total particle number is $N_e = L_x L_y$.

**Key design point:** The interleaved ordering $(\uparrow_1, \downarrow_1, \uparrow_2, \downarrow_2, \ldots)$ ensures that:
- Fermionic anticommutation relations are correctly handled by the Jordan-Wigner string (see §Fermionic Particle_Statistics)
- Translation symmetry acts naturally: both spin species at the same spatial site translate together
- The same `Symmetry_Operation` and `Finite_Symmetry_Group` infrastructure applies without modification

The example uses $[2,3]$ spatial unit cells (6 spatial sites → 12 graph vertices), with $N_\uparrow = N_\downarrow = 3$ ($N_e = 6$), and the $\mathbb{Z}_2 \times \mathbb{Z}_3$ translation group.

```bash
julia --project=. examples/fermion_hubbard_square.jl
```

**Key features demonstrated:**
- Flattening spin degrees of freedom into an interleaved graph
- Fermionic statistics (`Fermionic()`) with automatic Jordan-Wigner sign handling
- Construction of custom translation symmetry for spinful lattices
- Density-density Hubbard $U$ term connecting same-site up/down vertices

## Benchmarking

The `benchmark/` folder contains a comprehensive benchmarking suite that measures single-sector ED performance across all three model classes, comparing both matrix-construct and matrix-free modes.

### Running Benchmarks

The main benchmark script performs JIT warmup (running a small system twice in each mode to trigger compilation), then times exactly **one** symmetry sector for each model and system size. I test on my homeworkstation with **AMD EPYC 7K62 48-Core Processor**, with

```bash
julia --project=. -p 96 -t 96 benchmark/benchmark.jl
```

Results are written to CSV in `benchmark/results/` with timestamps. To generate plots from existing results:

```bash
julia --project=. benchmark/plot_benchmark.jl [optional_csv_path]
```

### Benchmark Coverage

| Model | System Sizes | Symmetry Group | Graph Vertices (max) |
|-------|-------------|----------------|---------------------|
| Spin-½ Heisenberg chain | $N = 20, 22, 24, 26, 28$ | $\mathbb{Z}_N$ | 28 |
| Bosonic Haldane FCI | $[2,3], [2,4], [2,5], [3,4], [2,7]$ | $\mathbb Z^{L_1}\times\mathbb Z^{L_2}$ | 28 |
| Spinful Fermi-Hubbard | $[2,3], [2,4], [2,5], [3,4], [2,7]$ | $\mathbb Z^{L_1}\times\mathbb Z^{L_2}$ | 28 |

To get a sense on the size of the benchmark
- the $N=28$ Heisenberg chain has a full Hilbert space of $2^{28}=2.68\times 10^8$ configurations, reduced by translation symmetry $|G|=28$ to sector dimensions of $9.58\times20^6$.
- the $[2,7]$ or $N=14$ spinful fermionic Hubbard model has dimension $C_{28}^{14}=4.01\times10^7$ configurations, but only reduced by translatoin symmetry with $|G|=14$ to sector dimensions $2.86\times 10^6$. 

### Output

Generated plots (saved to `benchmark/figures/`):
- **System-size plots** (`benchmark_<model>.svg`): elapsed time vs system size for both modes, with sector dimension annotated under each tick
- **Scaling plots** (`scaling_<model>.svg`): log-log plot of time vs sector Hilbert-space dimension, revealing the algorithmic scaling
- **Combined summary** (`benchmark_combined.svg`): all three models side-by-side in a single figure

The benchmark is designed to answer: *for a given model class and system size, should I use matrix or matrix-free mode?* The answer depends on the trade-off between memory (matrix-free wins) and speed (matrix mode is typically 1.5–3× faster for a single sector).

Benchmark Result:

![combined benchmark](../benchmark/figures/benchmark_combined.svg)

## Quick Reference

```bash
# ── Run examples ──────────────────────────────────────────────
julia --project=. examples/spin_heisenberg_chain.jl       # Heisenberg chain, N=20
julia --project=. examples/boson_fci_haldane.jl           # Bosonic FCI, 2×3 honeycomb
julia --project=. examples/fermion_hubbard_square.jl      # Fermi-Hubbard, 2×3 square

# ── Run benchmarks ───────────────────────────────────────────
julia --project=. -p 8 benchmark/benchmark.jl             # Full multi-model benchmark
julia --project=. benchmark/plot_benchmark.jl             # Plot from CSV (no re-run)

# ── Run tests ────────────────────────────────────────────────
julia --project=. test/runtests.jl                        # Sanity checks

# ── Plot ED spectrum from Julia REPL ─────────────────────────
using RealSpace_ExactDiagonalization
ed_data = ...                    # build or load your ED data
print_spectrum(ed_data)          # table of eigenvalues per sector
fig, ax = plot_spectrum(ed_data) # scatter plot (CairoMakie)
save("spectrum.svg", fig)
```

The design principles, full derivation, and API documentation are contained in this notebook. For usage examples and adaptation to custom models, copy from the `examples/` folder and modify the lattice construction and Hamiltonian terms.